Builds a filtered, balanced person/not-person subset from Wake Vision and exposes it as PyTorch DataLoaders.

In [ ]:
from datasets import load_dataset
import matplotlib.pyplot as plt
import os
from torchvision import datasets
from torchvision import transforms as T
from dotenv import load_dotenv
from torch.utils.data import DataLoader
import numpy as np

In [ ]:
# Route internet traffic through my local proxy (needed to reach Hugging Face).
# Set all six names because different libraries look for different ones.
proxy = "http://127.0.0.1:10808"
for var in ("HTTP_PROXY", "http_proxy", "HTTPS_PROXY", "https_proxy", "ALL_PROXY", "all_proxy"):
    os.environ[var] = proxy

In [ ]:
load_dotenv()
hf_token = os.environ.get("HF_TOKEN")

In [ ]:
USE_GRAYSCALE = False

## Dataset exploration

In [ ]:
ds = load_dataset("Harvard-Edge/Wake-Vision", streaming=True, split="train_quality", token=hf_token)

In [ ]:
ds.column_names

In [ ]:
ds = ds.select_columns(["image", "person"])
ds.column_names

In [ ]:
sample = next(iter(ds))
sample.items()

In [ ]:
(type(sample["image"]), type(sample["person"]))

In [ ]:
def show_raw_dataset(dataset_iterator, num_rows=3, num_columns=3):
  fig, axes = plt.subplots(num_rows, num_columns, figsize=(8, 5))

  for idx in range(num_rows * num_columns):
    sample = next(dataset_iterator)
    axes.flat[idx].imshow(sample["image"])
    axes.flat[idx].set_title(sample["person"])
    axes.flat[idx].axis("off")
  plt.show()

In [ ]:
it = iter(ds)
show_raw_dataset(it, 3, 5)

## Dataset creation

In [ ]:
def download_dataset(instance_per_class=300):
  ds = load_dataset("Harvard-Edge/Wake-Vision", streaming=True, split="train_quality", token=hf_token)
  # ds = ds.shuffle(seed=123, buffer_size=6000)

  # Collect an equal number of person / not_person images -> balanced dataset.
  # A balanced dataset avoids biasing the classifier toward
  # whichever class happens to be more common in Wake Vision.
  person_dir = "../data/subset/person"
  not_person_dir = "../data/subset/not_person"
  os.makedirs(person_dir, exist_ok=True)
  os.makedirs(not_person_dir, exist_ok=True)

  num_person = 0
  num_not_person = 0

  for idx, data in enumerate(ds):
      
      # Depicted images are not relevant in the project.
      #"person_depiction and non-person_non-depiction are -1 (unlabeled) in train_quality; filter on depiction instead" 
      if not data["depiction"]:

        # Wall mounted HLK-LD2410S <human presence sensor> only detects human
        # for up to 4 meters. so the far distance is not even trigger the
        # wake up sequence, thus not included.
        if(data["person"] and (data["medium_distance"] or data["near"]) and num_person < instance_per_class):
          img = data["image"].convert("L").resize((96, 96)) if USE_GRAYSCALE else data["image"].resize((96, 96))
          img.save(f"{person_dir}/{idx:06d}.png")
          num_person += 1

        elif(not data["person"] and num_not_person < instance_per_class):
          img = data["image"].convert("L").resize((96, 96)) if USE_GRAYSCALE else data["image"].resize((96, 96))
          img.save(f"{not_person_dir}/{idx:06d}.png")
          num_not_person += 1
        
      if num_person == instance_per_class and num_not_person == instance_per_class:
        print(f"#person: {num_person}, #not_person: {num_not_person}")
        break

In [ ]:
download_dataset()

In [ ]:
if USE_GRAYSCALE:
    transform = T.Compose([
        # ImageFolder will treat images as RGB images, even if they're stored as grayscale.
        T.Grayscale(),
        T.ToTensor(),
    ])
else:
    transform = T.Compose([
        T.ToTensor(),
    ])

data_dir = "../data/subset"
train_dataset = datasets.ImageFolder(data_dir, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)

In [ ]:
train_dataset.classes

In [ ]:
for imgs, labels in train_loader:
    print(f"input shape: {imgs.shape}, label shape: {labels.shape}")
    print(f"input dtype: {imgs[0].dtype}, label dtype: {labels[0].dtype}")
    print(f"label sample: {labels}")
    break

In [ ]:
def show_single_batch(dataset_loader, num_rows=2, num_columns=4):
  fig, axes = plt.subplots(num_rows, num_columns, figsize=(8, 5))
  
  for imgs, labels in train_loader:
    for idx in range(num_rows * num_columns):
      if USE_GRAYSCALE:
        axes.flat[idx].imshow(imgs[idx].numpy().reshape(96, 96), cmap="gray")
      else:
        axes.flat[idx].imshow(imgs[idx].numpy().transpose(1, 2, 0))
      axes.flat[idx].set_title(labels[idx])
      axes.flat[idx].axis("off")
    plt.show()
    break

In [ ]:
show_single_batch(train_loader)